### 1st-hour OSI, PIP, PEEP with TWs

In [1]:
# For V2
import os
import pandas as pd
from openpyxl import load_workbook

# === Input paths ===
csv_dir = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_RNN/V2_1st_Raw/OSI_V2_1st/TWs_V2_1st/PEEP_Settings_V2_1st/Abnormal_Deletion_V2_1st/TV_V2_1st"
excel_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_RNN/PARDS_Risk_V2_df.xlsx"

# === Read Sheet16 and prepare Sheet17 ===
df_excel = pd.read_excel(excel_path, sheet_name="Sheet16")
df_excel["FileName_V2_1st"] = df_excel["FileName_V2_1st"].astype(str).str.strip()
df_sheet17 = df_excel.copy()

# === Add output columns for each metric and TW ===
for metric in ["OSI", "PIP", "PEEP"]:
    for tw in range(1, 7):
        df_sheet17[f"{metric}_mean_TW{tw}"] = pd.NA
        df_sheet17[f"{metric}_std_TW{tw}"] = pd.NA

# Define column candidates
osi_col = "OSI"
pip_candidates = [
    "CAPSULE_AVEAA_VITAL_2564",
    "CAPSULE_DRAGERMEDIBUS_VITAL_2564",
    "CAPSULE_MAQUETC_VITAL_1414"
]
peep_candidates = [
    "CAPSULE_AVEAA_VITAL_1189",
    "CAPSULE_DRAGERMEDIBUS_VITAL_1189",
    "CAPSULE_MAQUETC_VITAL_1189"
]

# === Loop through CSVs ===
for filename in os.listdir(csv_dir):
    if not filename.endswith(".csv"):
        continue

    basename = filename.replace(".csv", "").strip()
    file_path = os.path.join(csv_dir, filename)
    df = pd.read_csv(file_path)

    match = df_sheet17[df_sheet17["FileName_V2_1st"] == basename]
    if match.empty or "Tumbling_window" not in df.columns or osi_col not in df.columns:
        continue

    pip_col = next((col for col in pip_candidates if col in df.columns and not df[col].dropna().empty), None)
    peep_col = next((col for col in peep_candidates if col in df.columns and not df[col].dropna().empty), None)

    for tw in range(1, 7):
        df_tw = df[df["Tumbling_window"] == tw]
        if df_tw.empty:
            continue

        # OSI
        osi_values = df_tw[osi_col].dropna()
        if not osi_values.empty:
            df_sheet17.loc[match.index, f"OSI_mean_TW{tw}"] = round(osi_values.mean(), 2)
            df_sheet17.loc[match.index, f"OSI_std_TW{tw}"] = round(osi_values.std(), 2)

        # PIP
        if pip_col:
            pip_values = df_tw[pip_col].dropna()
            if not pip_values.empty:
                df_sheet17.loc[match.index, f"PIP_mean_TW{tw}"] = round(pip_values.mean(), 2)
                df_sheet17.loc[match.index, f"PIP_std_TW{tw}"] = round(pip_values.std(), 2)

        # PEEP
        if peep_col:
            peep_values = df_tw[peep_col].dropna()
            if not peep_values.empty:
                df_sheet17.loc[match.index, f"PEEP_mean_TW{tw}"] = round(peep_values.mean(), 2)
                df_sheet17.loc[match.index, f"PEEP_std_TW{tw}"] = round(peep_values.std(), 2)

# === Save to Sheet17 ===
# book = load_workbook(excel_path)
with pd.ExcelWriter(excel_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    # writer.book = book
    df_sheet17.to_excel(writer, sheet_name="Sheet17", index=False)

print("✅ OSI, PIP, and PEEP mean/std by time window saved to Sheet17.")


✅ OSI, PIP, and PEEP mean/std by time window saved to Sheet17.


In [2]:
# For V1
import os
import pandas as pd

# === Input paths ===
csv_dir = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V1/VitalData/01_Vital_Raw_282_1st/OSI_285_1st/TWs_284_1st/PEEP_Settings_284_1st/Abnormal_Detection_282_1st/TV_V1_1st"
excel_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_RNN/PARDS_Risk_V1_df.xlsx"

# === Read Sheet10 and prepare Sheet11 ===
df_excel = pd.read_excel(excel_path, sheet_name="Sheet10")
df_excel["FileName_Vital_1st"] = df_excel["FileName_Vital_1st"].astype(str).str.strip()
df_sheet11 = df_excel.copy()

# === Add output columns for each metric and time window ===
for metric in ["OSI", "PIP", "PEEP"]:
    for tw in range(1, 7):
        df_sheet11[f"{metric}_mean_TW{tw}"] = pd.NA
        df_sheet11[f"{metric}_std_TW{tw}"] = pd.NA

# Define column names
osi_col = "OSI"
pip_candidates = ["AVEA - PIP", "CDGR - PIP", "SVU - PIP"]
peep_candidates = ["AVEA - PEEP", "CDGR - PEEP", "SVU - PEEP"]

# === Process each CSV ===
for filename in os.listdir(csv_dir):
    if not filename.endswith(".csv"):
        continue

    file_path = os.path.join(csv_dir, filename)
    df = pd.read_csv(file_path)

    match = df_sheet11[df_sheet11["FileName_Vital_1st"] == filename]
    if match.empty or "Tumbling_window" not in df.columns or osi_col not in df.columns:
        continue

    pip_col = next((col for col in pip_candidates if col in df.columns and not df[col].dropna().empty), None)
    peep_col = next((col for col in peep_candidates if col in df.columns and not df[col].dropna().empty), None)

    for tw in range(1, 7):
        df_tw = df[df["Tumbling_window"] == tw]
        if df_tw.empty:
            continue

        # OSI
        osi_values = df_tw[osi_col].dropna()
        if not osi_values.empty:
            df_sheet11.loc[match.index, f"OSI_mean_TW{tw}"] = round(osi_values.mean(), 2)
            df_sheet11.loc[match.index, f"OSI_std_TW{tw}"] = round(osi_values.std(), 2)

        # PIP
        if pip_col:
            pip_values = df_tw[pip_col].dropna()
            if not pip_values.empty:
                df_sheet11.loc[match.index, f"PIP_mean_TW{tw}"] = round(pip_values.mean(), 2)
                df_sheet11.loc[match.index, f"PIP_std_TW{tw}"] = round(pip_values.std(), 2)

        # PEEP
        if peep_col:
            peep_values = df_tw[peep_col].dropna()
            if not peep_values.empty:
                df_sheet11.loc[match.index, f"PEEP_mean_TW{tw}"] = round(peep_values.mean(), 2)
                df_sheet11.loc[match.index, f"PEEP_std_TW{tw}"] = round(peep_values.std(), 2)

# === Save to Sheet11 ===
with pd.ExcelWriter(excel_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    df_sheet11.to_excel(writer, sheet_name="Sheet11", index=False)

print("✅ OSI, PIP, and PEEP mean/std by time window saved to Sheet11.")


✅ OSI, PIP, and PEEP mean/std by time window saved to Sheet11.
